In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from src.spark_session import get_spark_session
from src.data_loader import load_raw_data_spark
from src.data_cleaning import clean_dataset_spark
from src.spark_analysis import *
from pyspark.sql import functions as F

spark = get_spark_session('Zomato_Analysis_Notebook')
raw_df = load_raw_data_spark(spark)
df, _ = clean_dataset_spark(raw_df, deduplicate=True)
df.cache()
print(f'Ready! Cleaned unique restaurants available: {df.count():,}')

In [ ]:
total_restaurants = df.count()
valid_ratings = df.filter(F.col('rating_num').isNotNull()).count()
print(f'Total Valid Restaurants: {total_restaurants:,}')
print(f'Restaurants with Customer Ratings: {valid_ratings:,} ({valid_ratings/total_restaurants*100:.1f}%)')

In [ ]:
locality_df = analyze_localities(df)
locality_df.show(10, truncate=False)

In [ ]:
cuisines_df = analyze_cuisines_popularity(df)
cuisines_df.show(10, truncate=False)

In [ ]:
ranked_cuisines = analyze_cuisine_ratings_with_threshold(df, min_restaurants=30)
ranked_cuisines.show(10, truncate=False)

In [ ]:
print('Rating Distribution:')
analyze_rating_distribution(df).show()
print('Cost Distribution by Price Tier:')
analyze_cost_distribution(df).show()

In [ ]:
correlations = analyze_cost_vs_rating_correlation(df)
print('Pearson Correlation Matrix:')
for k, v in correlations.items():
    print(f'  {k}: {v}')

In [ ]:
analyze_online_order_impact(df).show()

In [ ]:
analyze_table_booking_impact(df).show()

In [ ]:
high_eng = analyze_high_engagement_restaurants(df, top_n=10)
high_eng.select('name', 'location', 'rating_num', 'votes_num', 'engagement_score').show(10, truncate=False)

In [ ]:
import pandas as pd
from src.config import ANALYSIS_RESULTS_DIR
pd.read_csv(ANALYSIS_RESULTS_DIR / 'data_quality_report.csv')